<a href="https://colab.research.google.com/github/hyojun121/final/blob/suyeon/%EC%A4%91%EC%9A%94%ED%94%BC%EC%B2%98_20%EA%B0%9C_%2B_%ED%8A%9C%EB%8B%9D_(%ED%98%84%EC%9E%AC_%EA%B0%80%EC%9E%A5_%EB%86%92%EC%9D%8C).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# 기본
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# 경고 뜨지 않게 설정
import warnings
warnings.filterwarnings('ignore')

# 데이터 전처리 알고리즘
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import StandardScaler

# 학습용과 검증용으로 나누는 함수
from sklearn.model_selection import train_test_split

# 교차 검증
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import cross_validate
from sklearn.model_selection import KFold
from sklearn.model_selection import StratifiedKFold

# 평가함수
# 분류용
from sklearn.metrics import accuracy_score
from sklearn.metrics import precision_score
from sklearn.metrics import recall_score
from sklearn.metrics import f1_score
from sklearn.metrics import roc_auc_score

# 회귀용
from sklearn.metrics import r2_score
from sklearn.metrics import mean_squared_error

# 모델의 최적의 하이퍼 파라미터를 찾기 위한 도구
from sklearn.model_selection import GridSearchCV

# 머신러닝 알고리즘 - 분류
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import AdaBoostClassifier
from sklearn.ensemble import GradientBoostingClassifier
from lightgbm import LGBMClassifier as lgb
import xgboost as xgb

from sklearn.ensemble import VotingClassifier
from catboost import CatBoostClassifier


# 머신러닝 알고리즘 - 회귀
from sklearn.neighbors import KNeighborsRegressor
from sklearn.linear_model import LinearRegression
from sklearn.linear_model import Ridge
from sklearn.linear_model import Lasso
from sklearn.linear_model import ElasticNet
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import AdaBoostRegressor
from sklearn.ensemble import GradientBoostingRegressor
from lightgbm import LGBMRegressor
from xgboost import XGBRegressor
from sklearn.ensemble import VotingRegressor

# 학습 모델 저장을 위한 라이브러리
import pickle

In [ ]:
!pip install catboost

In [ ]:
df = pd.read_csv("/content/drive/MyDrive/open/workspace_suyeon/컬럼중요도20개 선택.csv")

In [ ]:
cat_cols = df.select_dtypes(include=['object', 'category']).columns.tolist()
cat_cols.remove('ID')
cat_cols

['Segment', '이용금액대']

In [ ]:
df.columns

Index(['ID', '기준년월', 'Segment', '카드이용한도금액', '정상청구원금_B0M', '입회경과개월수_신용',
       '이용금액_R3M_신용체크', '_1순위카드이용금액', '최대이용금액_일시불_R12M', '정상입금원금_B0M', '이용금액대',
       '월중평잔', '이용금액_일시불_B0M', '최대이용금액_할부_R12M', '이용금액_R3M_신용', '쇼핑_도소매_이용금액',
       '이용금액_오프라인_R6M', '이용금액_할부_R12M', '잔액_신판ca최대한도소진율_r6m', '최종이용일자_CA',
       '연체입금원금_B0M', '이용금액_CA_R12M', '최종이용일자_할부'],
      dtype='object')

In [ ]:
df2 = df.copy()

In [ ]:
# Segment 인코딩 사전
le_seg = LabelEncoder()
df2['Segment'] = le_seg.fit_transform(df2['Segment'])
seg_map = {cls: idx for idx, cls in enumerate(le_seg.classes_)}
print("Segment 인코딩 매핑 →", seg_map)

Segment 인코딩 매핑 → {'A': 0, 'B': 1, 'C': 2, 'D': 3, 'E': 4}


In [ ]:
# 문자열 -> 숫자
encoder_dict = {}
for col in cat_cols:  # cat_cols = ['방문횟수_PC_R6M', ...]
    le_col = LabelEncoder()
    df2[col] = le_col.fit_transform(df2[col])
    encoder_dict[col] = le_col  # 저장해둠

In [ ]:
df3 = df2.copy()

In [ ]:
# 입력과 결과로 나눈다
X = df3.drop(['ID','Segment'], axis = 1)
y = df3['Segment']

In [ ]:
# 스케일링

# 0) 원본 보존
X_scaled_all = X.copy()

# 1) 숫자형만 선택
num_cols = X.select_dtypes(include='number').columns

# 2) 스케일링
scaler = StandardScaler()
X_scaled_vals = scaler.fit_transform(X[num_cols])

# 3) 원본 DataFrame에 **덮어쓰기**
X_scaled_all[num_cols] = X_scaled_vals

# 🔹 이제 X_scaled_all 은 숫자형(표준화) + 범주형 그대로 포함



In [ ]:
train_X = X_scaled_all
train_y = y

In [ ]:
# 1️⃣ 데이터 분리
X_train, X_test, y_train, y_test = train_test_split(
    train_X, train_y, test_size=0.2, stratify=y, random_state=42)

### 모델

In [ ]:
# 2. 게이트 모델 (A/B vs CDE)
# ---------------------------------------------------------------
from lightgbm import LGBMClassifier, early_stopping
from scipy.optimize import brent
from sklearn.metrics import f1_score

ab_codes   = [seg_map['A'], seg_map['B']]
y_train_bin = y_train.isin(ab_codes).astype(int)
y_test_bin  = y_test.isin(ab_codes).astype(int)

lgb_bin = LGBMClassifier(
    device_type='gpu',
    class_weight='balanced',
    num_leaves=63, learning_rate=0.05, n_estimators=600,
    subsample=0.8, colsample_bytree=0.8, random_state=42
)

# ★ early_stopping을 callbacks 인자로 전달
lgb_bin.fit(
    X_train, y_train_bin,
    eval_set=[(X_test, y_test_bin)],
    eval_metric='binary_logloss',
    callbacks=[early_stopping(stopping_rounds=50, verbose=False)]
)

# ── (2-1) F1 최적 threshold 구하기
val_proba = lgb_bin.predict_proba(X_test)[:, 1]

def best_thr(p, y_true):
    return brent(lambda t: -f1_score(y_true, (p >= t).astype(int)),
                 brack=(0.1, 0.9))

thr_ab = best_thr(val_proba, y_test_bin)
print(f"🔑  Gate threshold = {thr_ab:.3f}")

# ── (2-2) 검증·추론 공통 마스크
mask_ab = val_proba >= thr_ab


[LightGBM] [Info] Number of positive: 893, number of negative: 1919107
[LightGBM] [Info] This is the GPU trainer!!
[LightGBM] [Info] Total Bins 4859
[LightGBM] [Info] Number of data points in the train set: 1920000, number of used features: 21
[LightGBM] [Info] Using GPU Device: Tesla T4, Vendor: NVIDIA Corporation
[LightGBM] [Info] Compiling OpenCL Kernel with 256 bins...
[LightGBM] [Info] GPU programs have been built
[LightGBM] [Info] Size of histogram bin entry: 8
[LightGBM] [Info] 20 dense feature groups (36.62 MB) transferred to GPU in 0.045136 secs. 1 sparse feature groups
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
[LightGBM] [Info] Start training from score 0.000000
🔑  Gate threshold = 0.597


In [ ]:
# ===============================================================
# 3. A/B 세부 분류 (CatBoost)
# ===============================================================
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTENC
from catboost import CatBoostClassifier

# (3-1) A/B 행만 추출
sel_ab_tr = y_train.isin(ab_codes)
X_ab_tr, y_ab_tr = X_train[sel_ab_tr], y_train[sel_ab_tr]

# (3-2) 범주형 위치 인덱스
cat_cols  = ['이용금액대']
cate_idx  = [X_ab_tr.columns.get_loc(c) for c in cat_cols]

# (3-3) SMOTE-NC 증식
smote_ab  = SMOTENC(categorical_features=cate_idx, random_state=42)
X_ab_res, y_ab_res = smote_ab.fit_resample(X_ab_tr, y_ab_tr)

# (3-4) 내부 검증 셋 분할  (A/B만 포함)
X_ab_train, X_ab_valid, y_ab_train, y_ab_valid = train_test_split(
    X_ab_res, y_ab_res,
    test_size=0.15, stratify=y_ab_res, random_state=42
)

# (3-5) CatBoost 학습
cat_ab = CatBoostClassifier(
    task_type='GPU', devices='0',
    iterations=800, depth=7, learning_rate=0.08,
    bagging_temperature=1, auto_class_weights='Balanced',
    random_state=42, early_stopping_rounds=50, verbose=False
)

cat_ab.fit(
    X_ab_train, y_ab_train,
    eval_set=(X_ab_valid, y_ab_valid)   # ← A/B만 있는 안전한 검증 셋
)

# 이제 cat_ab는 정상적으로 학습 완료!


In [ ]:
# ===============================================================
# 4. C/D/E 분류 (XGBoost)
# ===============================================================
cde_codes = [seg_map[c] for c in ['C', 'D', 'E']]

# --- (4-1) 학습용(Train) ---------------------------
sel_cde_tr = y_train.isin(cde_codes)          # ← 정답 기준 C/D/E 행
le_cde     = LabelEncoder()
y_cde_enc  = le_cde.fit_transform(y_train[sel_cde_tr])

# --- (4-2) 검증용(Valid) --------------------------
# 게이트 예측(~mask_ab) ∩ 정답도 C/D/E → "진짜 C/D/E"
# (필터링한 검증 마스크는 그대로 사용)
cde_mask_test = (~mask_ab) & y_test.isin(cde_codes)

xgb_cde = xgb.XGBClassifier(
    objective='multi:softprob', num_class=3,
    tree_method='gpu_hist', predictor='gpu_predictor',
    max_depth=9, min_child_weight=3, gamma=0.3,
    subsample=0.8, colsample_bytree=0.8,
    eta=0.06, reg_lambda=3, reg_alpha=1,
    n_estimators=800,                # ← 조기 종료 못 쓰니 적당히 늘리거나 줄여서
    random_state=42, verbosity=0
)

xgb_cde.fit(
    X_train[sel_cde_tr], y_cde_enc,
    eval_set=[(
        X_test[cde_mask_test],
        le_cde.transform(y_test[cde_mask_test])
    )],
    verbose=False            # ← 버그 없는 파라미터
)


XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.8, device=None, early_stopping_rounds=None,
              enable_categorical=False, eta=0.06, eval_metric=None,
              feature_types=None, gamma=0.3, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=None, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=9,
              max_leaves=None, min_child_weight=3, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=800,
              n_jobs=None, num_class=3, ...)

### 평가

In [ ]:
# 5. 검증 세트 F1 (0.90 도달 여부 체크)
# ===============================================================
# ─ (5-1) A/B 예측
prob_ab = cat_ab.predict_proba(X_test[ mask_ab ])
pred_ab = np.argmax(prob_ab, axis=1)          # 0:A, 1:B
code_ab = np.array([seg_map['A'], seg_map['B']])[pred_ab]

# ─ (5-2) C/D/E 예측
prob_cde   = xgb_cde.predict_proba(X_test[~mask_ab])
pred_cde_e = np.argmax(prob_cde, axis=1)
code_cde   = le_cde.inverse_transform(pred_cde_e)

# ─ (5-3) 통합
y_pred = np.empty(len(X_test), dtype=int)
y_pred[ mask_ab] = code_ab
y_pred[~mask_ab] = code_cde

macro_f1 = f1_score(y_test, y_pred, average="macro")
micro_f1 = f1_score(y_test, y_pred, average="micro")
print(f"🎯 Macro F1: {macro_f1:.4f}   |   Micro F1: {micro_f1:.4f}")

🎯 Macro F1: 0.7649   |   Micro F1: 0.8968


In [ ]:
test = pd.read_csv('/content/drive/MyDrive/open/Final_Project_Selected/TEST/all_test.csv')

In [ ]:
test1 = test[['ID', '기준년월', '카드이용한도금액', '정상청구원금_B0M', '입회경과개월수_신용',
       '이용금액_R3M_신용체크', '_1순위카드이용금액', '최대이용금액_일시불_R12M', '정상입금원금_B0M', '이용금액대',
       '월중평잔', '이용금액_일시불_B0M', '최대이용금액_할부_R12M', '이용금액_R3M_신용', '쇼핑_도소매_이용금액',
       '이용금액_오프라인_R6M', '이용금액_할부_R12M', '잔액_신판ca최대한도소진율_r6m', '최종이용일자_CA',
       '연체입금원금_B0M', '이용금액_CA_R12M', '최종이용일자_할부']]

In [ ]:
cat_cols = test1.select_dtypes(include=['object', 'category']).columns.tolist()
cat_cols.remove('ID')

In [ ]:
for col in cat_cols:
    test1[col] = encoder_dict[col].transform(test1[col])

In [ ]:
# 2) 숫자형 컬럼 표준화 (scaler 는 훈련 때 fit 된 객체)
test1[num_cols] = scaler.transform(test1[num_cols])

In [ ]:
# ──────────────────────────────────────────
# 3) Stage-1: A/B 후보 마스크
proba_ab = lgb_bin.predict_proba(test1.drop(['ID'], axis=1))[:, 1]
mask_ab  = proba_ab >= 0.5          # 학습 때 쓰던 threshold (필요 시 조정)

# ──────────────────────────────────────────
# 4) Stage-2 예측
pred = np.empty(len(test1), dtype=int)

#   4-A) A/B 부분
pred[mask_ab] = cat_ab.predict(test1.drop(['ID'], axis=1).iloc[mask_ab])

#   4-B) C/D/E 부분  (0·1·2 → 2·3·4 재변환)
cde_encoded = np.argmax(
    xgb_cde.predict_proba(test1.drop(['ID'], axis=1).iloc[~mask_ab]),
    axis=1
)
pred[~mask_ab] = le_cde.inverse_transform(cde_encoded)

# ──────────────────────────────────────────
# 5) 숫자 → 원본 Segment 문자 복원
inv_seg = {v: k for k, v in seg_map.items()}   # {0:'A',1:'B',...}
test1['Segment'] = [inv_seg[i] for i in pred]


In [ ]:
# 6) 제출 파일 저장
submission = pd.DataFrame({"ID": test1['ID'], "Segment": test1["Segment"]})
submission.to_csv("submission.csv", index=False)
print("✅  submission.csv 저장 완료!  행:", len(submission))

✅  submission.csv 저장 완료!  행: 100000
